# Field CBI validation: Shovel Creek and Nugget Creek

Santosh Panda supplied summer 2020 field records on October 2, 2026. This notebook audits the originals and evaluates the existing, **untuned** HLS severity measure at those plots. It does not refit recovery models or calibrate thresholds.

CBI is a field assessment on a 0-3 scale. We retain total scores as supplied, including valid zero scores. Smith et al. (2021), [DOI 10.3390/rs13101966](https://doi.org/10.3390/rs13101966), describes 30 m diameter plots and accessibility-based sampling. Exact plot survey dates were not supplied. Our HLS surface-reflectance summer composites differ from the paper's Sentinel-2 imagery and dates; this is a new comparison, not a replication of its accuracy figures.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').is_file())
sys.path.insert(0, str(REPO))
from scripts.validate_field_cbi import audit_fields, summarize, run, sha, CLASSES
ROOT = REPO.parent
FIELD_DIRECTORY = ROOT / 'Burn_severity_Survey_SHP'
RESULT_DIRECTORY = ROOT / 'validation/field_cbi_20261003'
REVIEW_DIRECTORY = ROOT / 'validation/field_cbi_review_20261003'
REVIEW_DIRECTORY.mkdir(exist_ok=True)

# Set True only to make a fresh live EE comparison; raw data remain local.
RUN_LIVE = False
if RUN_LIVE:
    from datetime import datetime, timezone
    RESULT_DIRECTORY = ROOT / 'validation' / ('field_cbi_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
    run(FIELD_DIRECTORY, RESULT_DIRECTORY, ROOT / 'validation/multi_fire_comparison')

## 1. Verify provenance and field scores

The source CRS is WGS84 UTM zone 6N (EPSG:32606). Plot names identify records; the numeric `Id` field is a placeholder. Shovel uses `CBI_T`, while Nugget uses `CBI`.

Missing stratum measurements are not zeros. Existing NBR/NDVI columns and `Training` flags come from earlier work and are excluded from our satellite comparison. The original shapefiles are preserved outside Git. Warnings below concern ancillary fields, not an automatic rejection of valid total CBI.

In [ ]:
manifest = json.loads((RESULT_DIRECTORY / 'manifest.json').read_text())
assert manifest['complete'] and not manifest['audit_only']
assert sha(REPO / 'scripts/validate_field_cbi.py') == manifest['script_sha256']
for module, digest in manifest['source_sha256'].items():
    assert sha(REPO / 'src' / module) == digest
for fire, digest in manifest['sample_manifests'].items():
    assert sha(ROOT / 'validation/multi_fire_comparison' / (fire.lower().replace(' ', '_') + '.json')) == digest
for file, digest in manifest['files'].items():
    assert sha(RESULT_DIRECTORY / file) == digest
plots, current_audit = audit_fields(FIELD_DIRECTORY)
audit = json.loads((RESULT_DIRECTORY / 'field_audit.json').read_text())
assert current_audit == audit
field_summary = pd.DataFrame([dict(fire=f, plots=a['n'], cbi_min=a['cbi_range'][0], cbi_max=a['cbi_range'][1], **a['class_counts']) for f, a in audit.items()])
display(field_summary)
for fire, a in audit.items():
    print(fire, 'warnings:', json.dumps(a['warnings']))
print('Total plots:', len(plots), '; exact survey dates unknown')

## 2. Verify the plot-to-pixel comparison

Use the existing June 15 inclusive to August 15 exclusive HLS composites for 2018 and 2020, on the explicit 30 m EPSG:32606 grid. dNBR = (NBR2018 - NBR2020) x 1000. Use common valid pre/post NBR and post NDVI pixels; no missing-data imputation.

The primary comparison averages satellite index values within a 15 m radius circle (30 m diameter), weighting intersecting pixel areas. A centre pixel and 20/30 m radius circles are sensitivity checks. Pixel counts are not independent observations and do not describe fractional-area weights. Field plots near the boundary remain eligible for extraction even if outside the mapped scar.

In [ ]:
data = pd.read_csv(RESULT_DIRECTORY / 'field_satellite.csv', float_precision='round_trip')
assert len(data) == len(plots) * 4
assert not data.duplicated(['fire', 'plot_id', 'mode']).any()
assert set(data['mode']) == {'center', 'plot_15m', 'buffer_20m', 'buffer_30m'}
joined = data.merge(plots[['fire', 'plot_id', 'cbi']], on=['fire', 'plot_id'], suffixes=('', '_raw'), validate='many_to_one')
np.testing.assert_array_equal(joined.cbi, joined.cbi_raw)
valid = data.loc[data.dnbr_mean.notna()]
assert (valid.dnbr_count > 0).all()
# EE index arithmetic uses float32; 1e-4 scaled units covers its rounding.
np.testing.assert_allclose(valid.dnbr_mean, (valid.nbr_2018_mean - valid.nbr_2020_mean) * 1000, rtol=0, atol=1e-4)
assert valid[['nbr_2018_mean', 'nbr_2020_mean', 'ndvi_2020_mean']].abs().le(1).all().all()
assert valid[['inputs_2018_mean', 'inputs_2020_mean']].gt(0).all().all()
display(data.groupby(['fire', 'mode']).agg(plots=('plot_id', 'size'), valid=('dnbr_mean', 'count'), inside=('inside_perimeter', 'sum'), min_2018_inputs=('inputs_2018_mean', 'min'), min_2020_inputs=('inputs_2020_mean', 'min')))
print('Index arithmetic, score linkage, coverage and source fingerprints verified.')

## 3. Evaluate association and provisional classes

Spearman rank correlation asks whether higher field severity generally accompanies larger satellite change. It does not establish an accurate class threshold or explain recovery. Report each fire separately; pooled results could reflect differences between fires. Do not use ordinary correlation p-values for these clustered plots.

For the baseline class comparison, field classes are: zero = unburned; above zero to below 1.5 = low; 1.5 to below 2 = moderate; 2-3 = high. The existing dNBR bins are untuned. Merge the two moderate bins, keeping values below -100 in a separate increased-greenness category. A high-CBI plot is CBI >= 2; satellite high is dNBR > 660. Class agreement applies to these selected plots, not the fire's area.

In [ ]:
summary = json.loads((RESULT_DIRECTORY / 'summary.json').read_text())
assert summarize(data) == summary
rows = []
for record in summary:
    subset = data.loc[(data.fire == record['fire']) & (data['mode'] == record['mode']) & data.dnbr_mean.notna()]
    independent_rho = subset.cbi.rank().corr(subset.dnbr_mean.rank())
    np.testing.assert_allclose(independent_rho, record['spearman']['dnbr_mean'], rtol=0, atol=1e-14)
    assert np.asarray(record['confusion_counts']).sum() == record['n_valid']
    h = record['high_class']
    rows.append(dict(fire=record['fire'], mode=record['mode'], n=record['n_valid'], rho_dnbr=independent_rho,
        class_agreement=record['fixed_class_agreement'], high_detected=h['tp'], high_missed=h['fn'], false_high=h['fp']))
display(pd.DataFrame(rows).round(4))
primary = [r for r in summary if r['mode'] == 'plot_15m']
for r in primary:
    print(r['fire'])
    display(pd.DataFrame(r['confusion_counts'], index=r['confusion_rows'], columns=r['confusion_columns']).rename_axis(index='Field CBI', columns='HLS dNBR'))

In [ ]:
burned_rows = []
for record in primary:
    group = data.loc[(data.fire == record['fire']) & (data['mode'] == 'plot_15m')]
    burned = group.loc[group.cbi > 0]
    burned_rows.append(dict(fire=record['fire'], all_plots=len(group), burned_plots=len(burned),
        rho_all=group.cbi.rank().corr(group.dnbr_mean.rank()),
        rho_burned=burned.cbi.rank().corr(burned.dnbr_mean.rank()),
        rho_pre_nbr=group.cbi.rank().corr(group.nbr_2018_mean.rank())))
display(pd.DataFrame(burned_rows).round(4))
print('Including unburned plots strengthens the overall association. Keep the burned-only diagnostic visible.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
for row, record in enumerate(primary):
    fire = record['fire']
    group = data.loc[(data.fire == fire) & (data['mode'] == 'plot_15m')]
    ax = axes[row, 0]
    for label, color in zip(CLASSES, ['#668567', '#acb55b', '#c89853', '#975653']):
        part = group.loc[group.cbi_class == label]
        ax.scatter(part.cbi, part.dnbr_mean, color=color, label=label, edgecolors='white', linewidths=.5, s=48)
    ax.axhline(660, color='#777777', linestyle='--', linewidth=1, label='Current high cutoff')
    ax.axvline(2, color='#777777', linestyle=':', linewidth=1)
    ax.set(xlim=(-.08, 3.05), ylim=(-150, 1100), xlabel='Field CBI (0-3)', ylabel='HLS dNBR x1000',
           title=f"{fire}: n={len(group)}, rank correlation={record['spearman']['dnbr_mean']:.3f}")
    ax.legend(fontsize=8, loc='upper left')
    matrix = np.asarray(record['confusion_counts'])
    ax = axes[row, 1]
    ax.imshow(matrix, cmap='YlGn', vmin=0, vmax=14)
    ax.set(xticks=range(5), xticklabels=['Unburned', 'Low', 'Moderate', 'High', 'Increased\ngreenness'],
           yticks=range(4), yticklabels=CLASSES, xlabel='HLS class (untuned)', ylabel='Field CBI class',
           title=f"Selected-plot class agreement: {record['fixed_class_agreement']:.1%}")
    for i in range(4):
        for j in range(5):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center', color='white' if matrix[i, j] > 8 else '#263c34')
figure_path = REVIEW_DIRECTORY / 'cbi_comparison.png'
fig.savefig(figure_path, dpi=160)
plt.show()
verification = dict(checked_files=len(manifest['files']), total_plots=len(plots), satellite_rows=len(data),
    primary_valid=sum(r['n_valid'] for r in primary), independent_rank_correlations=True,
    dnbr_identity_atol=1e-4, figure_sha256=sha(figure_path), result_manifest_sha256=sha(RESULT_DIRECTORY / 'manifest.json'))
(REVIEW_DIRECTORY / 'verification.json').write_text(json.dumps(verification, indent=2) + '\n')

## Interpretation and the next decision

- All 80 plots have valid satellite values in every tested footprint and lie inside the selected fire perimeters. Their accessible, clustered locations limit geographic inference.
- All-plot rank correlations are about 0.805/0.876 for Shovel/Nugget. Burned-only correlations are lower, about 0.598/0.657. Distinguishing unburned from burned contributes to the overall association. Footprint sensitivity does not reverse the overall ordering.
- Generic class cutoffs miss many high-CBI plots. Good rank correlation and poor class sensitivity can coexist. Do not label the current map field-calibrated.
- The raw `Ove_B = 1000` entry at Shovel SC21 is flagged; its supplied total CBI is retained. Shovel SC1 has few recorded stratum averages. Nugget total scores and class labels agree. These warnings should be clarified before attempting stratum-specific calibration.
- Exact survey-day timing remains unknown. Paper dates, sensor processing and sampling differ from ours, so published accuracy percentages are not comparison targets.
- Next: freeze this baseline, examine vegetation/background effects, then assess a simple calibration using a fire held out from fitting. Report both transfer directions and buffer sensitivities. With only two field fires, broad Alaska generalization remains untested.
- CBI measured in 2020 provides burn-severity reference evidence. It does not validate five-year forest recovery, soil recovery or the NDVI slope models.

Stack: Python, PyShp, pandas/NumPy/SciPy, Earth Engine, Matplotlib, Jupyter. Sources used: Santosh Panda's supplied CBI shapefiles/cheat sheet; NASA HLS L30/S30 v002; Alaska Fire History shared asset; Smith et al. (2021) for plot design and class definitions. Generated records and figures stay local.